In [ ]:
import pandas as pd
import MySQLdb
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)
import json
import urllib.request
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn')
""
import psycopg2 as pg

import matplotlib.pyplot as plt
#plt.style.use("ggplot")
import seaborn as sns

In [ ]:
gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x).date()
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)




In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py




db = client.get_database('telematics_db')




db_iten = client_iten.get_database('telematics_db')







In [ ]:
device_health_complete=pd.read_sql("select * from device_health",con=engine)

In [ ]:
device_health_complete[device_health_complete['vehicle_id']==100001]

In [ ]:
dh_no_dup=device_health_complete.drop_duplicates(['vehicle_id','day'])


In [ ]:
dh_no_dup_raw_pings=dh_no_dup[dh_no_dup['total_raw_pings']>0]

In [ ]:
dh_no_dup_raw_pings.day.value_counts()

In [ ]:
blacklist_devices=pd.read_csv('blacklist_devices.csv')

In [ ]:
len(blacklist_devices['vehicle_id'].unique())

In [ ]:
dh_no_dup_raw_pings.head(1)

In [ ]:
plt.hist(dh_no_dup_raw_pings[dh_no_dup_raw_pings['vehicle_id'].isin(blacklist_devices['vehicle_id'])].groupby('vehicle_id')['jump_percent'].mean(),bins=range(0,100,1))
plt.xlabel("jump percentage avg ")
plt.ylabel("frequency out of 3800 blacklisted devices")

In [ ]:
blacklist_health=dh_no_dup_raw_pings[dh_no_dup_raw_pings['vehicle_id'].isin(blacklist_devices['vehicle_id'])]

In [ ]:
plt.hist(blacklist_health['jump_percent'],bins=range(-2,100,1))
plt.ylabel("frequency out of 3800*5 unique combination of day & device")
plt.xlabel("jump percent per day (3800*5 unique days and vehicles ka combination)")

In [ ]:
blacklist_health['jump_percent'].describe(percentiles=[0.2,0.25,0.3,0.4,0.6,0.7,0.8,0.9,0.92,0.94,0.96,0.98])

In [ ]:
vehicle_details=pd.read_sql("select vehicle_id,model_name from vehicle_details where vehicle_state='LIVE'",con=engine)

In [ ]:
grids = sns.PairGrid(blacklist_health[['total_raw_pings','usedSat_mean','jump_percent','expected_run_pings_percent','cnsctv_time_diff_mean']],vars=['total_raw_pings', 'jump_percent','expected_run_pings_percent','cnsctv_time_diff_mean']) 

grids = grids.map_diag(plt.hist,color='red')
grids = grids.map_offdiag(plt.scatter,alpha=0.1,edgecolor="w")

In [ ]:
j=[]
for i in list(dh_no_dup_raw_pings.model_name.unique()):
    
    try:
        percentage=100*len(blacklist_health[blacklist_health['model_name']==i]['vehicle_id'].unique())/len(vehicle_details[vehicle_details['model_name']==i])
    except:
        percentage=None
    count=len(blacklist_health[blacklist_health['model_name']==i]['vehicle_id'].unique())
    j.append([i,percentage,count])

In [ ]:
percentages=pd.DataFrame(j,columns=['model_name','percentage','count'])

In [ ]:
device_history=pd.DataFrame()
for i in list(blacklist_devices['vehicle_id'].unique()):
    device_history_reuse=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':int(i),'toTime':{'$gte':tm.time()-8*86400}})))
    device_history=pd.concat([device_history,device_history_reuse])

In [ ]:
device_history

In [ ]:
no_info=[]
for i in list(blacklist_devices['vehicle_id'].unique()):
    total_no_info=len(device_history[device_history['vehicleId']==i])
    wire_cut=len(device_history[(device_history['vehicleId']==i)&(device_history['suggestion']=='Battery disconnected')])
    no_info.append([i,total_no_info,wire_cut])
    


In [ ]:
no_info_pd=pd.DataFrame(no_info,columns=['vehicle_id','total_no_info','wire_cuts'])

In [ ]:
plt.hist(no_info_pd['wire_cuts'],bins=range(0,10,1))
plt.ylabel('frequency of vehicles (blacklisted 3800)')
plt.xlabel('number of no infos in last 7 days per vehicle')

In [ ]:
device_history[(device_history['vehicleId']==100001)&(device_history['fromTime']==1564469181)]

In [ ]:
percentages.sort_values('percentage',ascending=False)

In [ ]:
dh_no_dup_raw_pings_vid_group=dh_no_dup_raw_pings.groupby('vehicle_id')

In [ ]:
more_jump=dh_no_dup_raw_pings_vid_group.jump_percent.mean()

In [ ]:
vehids=(dh_no_dup_raw_pings_vid_group.jump_percent.mean()>20).keys()

In [ ]:
a=pd.DataFrame()

In [ ]:
a['vehicle_id']=vehids

In [ ]:
a['jump_percent']=more_jump

In [ ]:
a['jump_percent'].isnull().value_counts()

In [ ]:
no_data=device_health[device_health['cnsctv_time_diff_mean']==-1]

In [ ]:
grids = sns.PairGrid(device_health[device_health['model_name']=='C100'][['total_raw_pings','usedSat_mean','jump_percent','expected_run_pings_percent','cnsctv_time_diff_mean']], vars=['total_raw_pings', 'jump_percent','expected_run_pings_percent','cnsctv_time_diff_mean']) 

grids = grids.map_diag(plt.hist,color='red')
grids = grids.map_offdiag(plt.scatter,alpha=0.1,edgecolor="w")




In [ ]:
device_health=device_health[device_health['cnsctv_time_diff_mean']!=-1]

In [ ]:
device_health.drop_duplicates(['day','vehicle_id'],inplace=True)

In [ ]:
len(device_health)

In [ ]:
plt.hist(no_data['vehicle_id'],bins=1000)
plt.xlabel('range of vehicle IDs')
plt.ylabel('no. of vehicles with no data')

In [ ]:
dt.datetime.today()

In [ ]:
device_health.head(1)

In [ ]:
device_health[['jump_percent']].describe(percentiles=[0.8,0.85,0.90,0.91,0.92,0.93,0.94,0.95,0.96,0.97,0.98,0.99])

In [ ]:
device_health[device_health['jump_percent']>23]['model_name'].value_counts()

In [ ]:
device_health['jump_percent'].plot.hist

In [ ]:
device_health['jump_percent'].plot.hist?

In [ ]:
device_health['jump_percent'].plot.hist?

In [ ]:
device_health['jump_percent'].plot.hist

In [ ]:
a=pd.DataFrame(list(db.deviceHistory.find({'fromTime':{'$gte':(tm.time()-86400)}})))

In [ ]:
a[a['suggestion']=='Battery disconnected'][a['toTime'].notnull()]

In [ ]:
2031-1184

In [ ]:
a=pd.read_sql("select count(*) from operator_vehicles where deboarding_time is null",con=mysql_vds)

In [ ]:
a['count(*)'].iloc[0]

In [ ]:
dh_no_dup

In [ ]:
dt.datetime.now().time?

In [ ]:
strftime(dt.datetime(),'%Y')

In [ ]:
%%time
dt.datetime.now().strftime('%H:%M:%S:%s')


In [ ]:
for i in range(900,1,-100):
    print(i)